In [ ]:
import xml.etree.ElementTree as et
import pandas as pd
import re
import os
from iso3166 import countries
from src.xml_parse import get_country, xml_get_text, xml_parse_baseline

In [ ]:
xml_file = '../data/PMC012xxxxxx/PMC12005907.xml'
xtree = et.parse(xml_file, parser=et.XMLParser(encoding="UTF-8"))
xroot = xtree.getroot()

In [ ]:
for child in xroot: 
    print(child.tag, child.attrib)
    for child2 in child:
        print('    ', child2.tag, child2.attrib)
        for child3 in child2:
            print('        ', child3.tag, child3.attrib)

In [ ]:
xroot.tag

In [ ]:
secs = xroot.findall('./body/sec')
len(secs)

In [ ]:
xml_get_text([secs[1]])

In [ ]:
titles = xroot.findall('./body/sec/title')
len(titles)

In [ ]:
def get_sec_titles(root):
    titles = root.findall('./body/sec/title')
    title_count = len(titles)
    
    title_names = [title.text for title in titles]

    return title_count, title_names

In [ ]:
title_count, title_names = get_sec_titles(xroot)

In [ ]:
title_count

In [ ]:
xml_get_attr(xroot, '{http://www.w3.org/XML/1998/namespace}lang')

In [ ]:
xml_get_attr(xroot, 'article-type')

In [ ]:
xml_get_text(xroot.findall('./front/journal-meta/journal-title-group/journal-title'))

In [ ]:
# not all articles have pmid, so also collect pmc id
xml_get_text(xroot.findall('./front/article-meta/article-id/[@pub-id-type="pmid"]'))

In [ ]:
xml_get_text(xroot.findall('./front/article-meta/article-id/[@pub-id-type="pmc"]'))

In [ ]:
xml_get_text(xroot.findall('./front/article-meta/title-group/article-title'))

In [ ]:
countries.get('brazil')

In [ ]:
xroot.findall('./front/article-meta/contrib-group/aff/country')[0].text

In [ ]:
validate_country('Republic of Korea')

In [ ]:
get_country(xroot)

In [ ]:
try:
    c = countries.get('Us').apolitical_name
except KeyError:
    c = None

c

In [ ]:
# attributes to add @pub-type="ppub", 
def get_date(root):
    date = xml_get_text(root.findall('./front/article-meta/pub-date/[@pub-type="epub"]'), '-')

    if date == None:
        date = xml_get_text(root.findall('./front/article-meta/pub-date/[@date-type="pub"]'), '-')

    return date

In [ ]:
get_date(xroot)

In [ ]:
# alternative version: get date dict to include tags to make sure day and month aren't confused
nodes = xroot.findall('./front/article-meta/pub-date/[@pub-type="epub"]')
date = {}
if len(nodes) > 0:
    for child in nodes[0]:
        date[child.tag] = child.text
date

In [ ]:
# some abstracts are structured into background/methods/results, 
# the section titles are part of tha abstracts as of now.
# How to remove?

def xml_get_abstr(node):
    abs = ""
    for a in node:
        if 'graphical' in a.attrib.values():
            continue 
        abs += xml_get_text(node)
    
    return abs

In [ ]:
xml_get_abstr(xroot.findall('./front/article-meta/abstract'))

In [ ]:
xml_parse_single(xml_file)

In [ ]:
baseline_df = xml_parse_baseline('../data', '../data/baseline_PMC12.json')